# Creating a Simple Agent with Tracing

In [5]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [10]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [11]:
placement_trainer_agent = Agent(
    name="Placement Trainer",
    instructions="""
    You are a helpful placement preparation trainer.
    Help users prepare for aptitude tests, coding interviews, technical interviews,
    behavioral interviews, resumes, and placement planning.
    Give concise, practical answers with clear next steps.
    Ask one focused question when important context is missing.
    Do not invent company-specific requirements; state when details may vary.
    """,
)

Let's execute the Agent:

In [15]:
with trace("Placement Trainer Agent"):
    result = await Runner.run(
        placement_trainer_agent,
        "Create a concise 7-day plan for preparing for a software placement interview.",
    )

print(result)

RunResult:
- Last agent: Agent(name="Placement Trainer", ...)
- Final output (str):
    Here’s a concise 7-day plan you can follow (roughly 2–3 hours per day). Adjust the timing to fit your schedule.
    
    Day 1 — Baseline and resume/collect
    - Take a 20–30 min self-assessment: list 5 strengths, 3 projects, 2 notable outcomes.
    - Create/update resume focusing on impact, quantify results.
    - Gather 5–6 real coding problems you’ve solved (or plan to solve) and note topics.
    - Quick system-design mental map: know basic components of a typical system (DB, cache, APIs, load balancer, etc.).
    
    Day 2 — Data structures fundamentals (core 1)
    - Review arrays, linked lists, stacks, queues, trees, graphs.
    - Practice 4–6 problems focused on these topics (LeetCode easy–medium).
    - Learn complexity analysis and common patterns (two-pointers, sliding window, DFS/BFS).
    
    Day 3 — Algorithms patterns (core 2)
    - Focus on sorting, binary search, DP basics, greedy

Streaming the answer to the screen, token by token

In [13]:
response_stream = Runner.run_streamed(
    placement_trainer_agent,
    "Give me three concise tips for a technical placement interview.",
)

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

- Practice with timed coding problems focusing on common DS/Algo patterns (arrays, strings, linked lists, trees/graphs, DP, hashing).  
- Think aloud: state your plan, discuss time/space complexity, edge cases, and possible optimizations as you code.  
- Ask clarifying questions and confirm constraints before you start; be ready to adapt if requirements change.

In [17]:
with trace("Placement Trainer Agent"):
    result = await Runner.run(
        placement_trainer_agent,
        "Create a concise 7-day plan for preparing for a software placement interview.",
    )

print(result)

RunResult:
- Last agent: Agent(name="Placement Trainer", ...)
- Final output (str):
    Here’s a concise 7-day plan to prepare for a software placement interview.
    
    Day 1 — Understanding & Foundations
    - Goals: understand process, gather materials, set up.
    - Tasks:
      - Review typical interview flow: aptitude, data structures/algorithms, coding, system design basics, behavioral.
      - Compile a personal resume snapshot and 2–3 projects to discuss.
      - Gather resources: 2–3 preferred problem sets (e.g., LeetCode Easy–Medium), mock interview platform, a notebook.
    - Output: target list of 5–6 topics to focus on; 2–3 example questions you’ll discuss.
    
    Day 2 — Data Structures Refresh (Part 1)
    - Goals: solidify core DS concepts.
    - Tasks:
      - Review arrays, strings, linked lists, stacks/queues, hash maps.
      - Solve 6–8 quick problems (2–3 per DS) with focus on time/space complexity.
      - Note patterns: sliding window, two-pointers, fast/sl

_Good Job!_